# Fase 2 — Coleta (Bronze)

Coleta dados brutos de PNAD, CAGED, RAIS e Censo.
Cada collector é idempotente: arquivos existentes são pulados.

> **Autenticação BigQuery:** na primeira execução o `basedosdados` abrirá
> um link de autorização OAuth no navegador. Siga as instruções e cole o
> código de volta no prompt.

In [ ]:
import sys, pathlib

_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
from config import config

print(f"GCP project      : {config.gcp_project_id}")
print(f"UFs              : {config.pnad_uf_siglas}")
print(f"Período          : {config.ano_inicio} – {config.ano_fim}")
print(f"CAGED modo teste : {config.caged_ano_teste or 'desativado (coleta completa)'}")
print(f"RAIS  modo teste : {config.rais_ano_teste  or 'desativado (coleta completa)'}")
print(f"BQ limite teste  : {f'{config.bq_limite_teste:,} linhas' if config.bq_limite_teste else 'desativado (todas as linhas)'}")
print(f"Bronze dir       : {config.bronze_dir}")

In [ ]:
from coleta import PnadCollector

print("=== Fase 2.1 — PNAD ===")
pnad = PnadCollector(config.bronze_dir, config.pnad_uf_codes)
pnad.collect()

In [ ]:
from coleta import CagedCollector

print("=== Fase 2.2 — CAGED ===")
caged = CagedCollector(
    config.bronze_dir,
    config.gcp_project_id,
    config.pnad_uf_siglas,
    config.ano_inicio,
    config.ano_fim,
    ano_teste=config.caged_ano_teste,
    limite_teste=config.bq_limite_teste,
)
caged.collect()

In [ ]:
from coleta import RaisCollector

print("=== Fase 2.3 — RAIS ===")
rais = RaisCollector(
    config.bronze_dir,
    config.gcp_project_id,
    config.pnad_uf_siglas,
    config.ano_inicio,
    config.ano_fim,
    ano_teste=config.rais_ano_teste,
    limite_teste=config.bq_limite_teste,
)
rais.collect()

In [ ]:
from coleta import CensoCollector

print("=== Fase 2.4 — Censo 2022 ===")
censo = CensoCollector(config.bronze_dir, config.pnad_uf_codes)
censo.collect()